# MaaS validation demo (pre-provisioned API key)

Same flow as **`maas-validation-demo.ipynb`**, but **skips OpenShift login and API key creation**. You supply a **MaaS API key** you created elsewhere.

**RHOAI demos:** Use **Demo quick swap** below to paste `MAAS_BASE` and your API key between runs. See **README → OpenShift CLI and console** for `oc` / console URLs (you typically do **not** run `oc` inside the notebook).

**Prerequisites:** Python 3.9+ (stdlib only — `urllib`, `ssl`; no `pip install` required).

## Demo quick swap

Edit the **next cell** only. Re-run **that cell** + **Setup** after each change.

| Variable | Use |
|----------|-----|
| `DEMO_MAAS_BASE` | Gateway URL, no trailing slash. `""` → use `MAAS_BASE` from the environment. |
| `DEMO_API_KEY` | MaaS API key string. `""` → use `MAAS_API_KEY` or `API_KEY` from the environment. |

Non-empty `DEMO_*` values **override** the environment.

In [ ]:
DEMO_MAAS_BASE = ""
DEMO_API_KEY = ""

## Setup

Loads configuration, defines **`http_json`**, prints a short summary (no secrets).

In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request
from typing import Any, Dict, Optional

_mb = globals().get("DEMO_MAAS_BASE", "")
if isinstance(_mb, str) and _mb.strip():
    MAAS_BASE = _mb.strip().rstrip("/")
else:
    MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE").strip().rstrip("/")

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")
MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"

if not API_KEY:
    raise SystemExit(
        "Set DEMO_API_KEY in the quick-swap cell or MAAS_API_KEY / API_KEY in the environment."
    )


def http_json(
    method: str,
    url: str,
    *,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
):
    """Minimal JSON HTTP helper (stdlib only)."""
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = None
    if data is not None:
        body = json.dumps(data).encode("utf-8")
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


print("MAAS_BASE :", MAAS_BASE)
print("MODELS_URL:", MODELS_URL)
print("VERIFY_TLS:", VERIFY_TLS)
print("API key set:", bool(API_KEY))

## 1. Model list (discovery)

`GET /maas-api/v1/models` with the API key returns models for that subscription. We capture **`MODEL_NAME`** (`id`) and **`MODEL_URL`** for the next steps.

In [ ]:
_, models_body = http_json("GET", MODELS_URL, token=API_KEY)
print(json.dumps(models_body, indent=2)[:4000])

data = models_body.get("data") or []
if not data:
    raise SystemExit("No models in response; deploy a model and check subscription binding.")

first = data[0]
MODEL_NAME = first.get("id") or first.get("name")
MODEL_URL = (first.get("url") or "").rstrip("/")
if not MODEL_NAME or not MODEL_URL:
    raise RuntimeError(f"Could not parse model id/url from: {first}")

print("\nUsing MODEL_NAME:", MODEL_NAME)
print("Using MODEL_URL :", MODEL_URL)

## 2. Single inference call

One request to **`{MODEL_URL}/v1/completions`**. Adjust payload if your model expects chat format.

In [ ]:
COMPLETIONS_URL = f"{MODEL_URL}/v1/completions"
inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Hello from the notebook demo.",
    "max_tokens": 50,
}

status, completion = http_json("POST", COMPLETIONS_URL, token=API_KEY, data=inference_payload)
print("HTTP", status)
print(json.dumps(completion, indent=2)[:4000])

## 3. Repeat calls until HTTP 429

Sends the same style of request, reusing **`MODEL_URL`** / **`MODEL_NAME`**. Stop on first **429** or after **`MAX_REQUESTS`** attempts.

In [ ]:
MAX_REQUESTS = 64
inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Rate limit probe.",
    "max_tokens": 50,
}

ctx = ssl.create_default_context()
if not VERIFY_TLS:
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

last_code = None
for i in range(1, MAX_REQUESTS + 1):
    body = json.dumps(inference_payload).encode("utf-8")
    req = urllib.request.Request(
        COMPLETIONS_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            last_code = resp.status
    except urllib.error.HTTPError as e:
        last_code = e.code
    print(f"{i:3d}  HTTP {last_code}")
    if last_code == 429:
        print("Got 429 — rate limit enforced.")
        break
else:
    print(f"No 429 after {MAX_REQUESTS} requests (quota may be high or window large).")